# 商業洞察（商業分析師）

讀 `input/cleaned.csv`，依最近一期還款狀況（PAY_0）分四群。

In [ ]:
import pandas as pd
import json
from pathlib import Path

Path("output").mkdir(exist_ok=True)
df = pd.read_csv("input/cleaned.csv")
overall = df["default"].mean()
print("整體違約率", round(overall, 4), "樣本", len(df))

In [ ]:
bands = pd.cut(df["PAY_0"], bins=[-100, 0, 1, 2, 100], labels=["準時", "遲繳1期", "遲繳2期", "遲繳3期以上"])
seg = df.groupby(bands, observed=True)["default"].agg(["count", "mean"]).reset_index()
seg.columns = ["segment", "n", "default_rate"]
seg["default_rate"] = seg["default_rate"].round(4)
seg

In [ ]:
top = seg.sort_values("default_rate", ascending=False).iloc[0]
report = {
    "segment_by": "PAY_0_BAND",
    "overall_default_rate": round(float(overall), 4),
    "segments": [{"segment": str(r.segment), "n": int(r.n), "default_rate": float(r.default_rate)} for r in seg.itertuples()],
    "highest_risk_segment": str(top.segment),
    "insight": f"{top.segment} 的違約率 {top.default_rate:.1%}，是整體 {overall:.1%} 的 {top.default_rate / overall:.1f} 倍；還款延遲越久違約率越高，授信應優先看 PAY_0。",
}
Path("output/insight_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))

## 給客戶的結論

整體違約率 25.2%。**還款延遲越久，違約率越高**：準時繳款的客戶違約率 12.1%（1631 人），遲繳 1 期 35.0%（460 人），遲繳 2 期 63.1%（298 人），遲繳 3 期以上 75.2%（109 人）。最高風險的一群是整體的 3 倍。

## 建議行動與影響範圍

建議把「最近一期遲繳 2 期以上」設為人工複審門檻。這一群共 407 人，占循環戶 16.3%，但其中約 272 人會違約，占全部違約案件的 43%。先管住這 16% 的客戶，就能攔下四成多的壞帳。

## 不確定的地方

1. 遲繳 3 期以上只有 109 人，違約率 75.2% 的誤差範圍大約正負 8 個百分點。
2. PAY_0 有 75 筆是資料工程補的中位數，這些人被歸到「準時」附近，準時群的 12.1% 可能略高估。
3. 這是相關不是因果：遲繳是違約的前兆，不代表催收遲繳就一定降低違約。